# Querying with SQL & Python — Chinook Business Analysis

**Objective:** Connect Python to SQLite, inspect the Chinook schema, answer business questions with SQL, and communicate findings using Pandas and Matplotlib.

> Run the notebook from top to bottom. The database is downloaded automatically if it is not already present in this folder.

## 1. Import libraries

In [ ]:
import sqlite3
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

## 2. Get the Chinook database

The database is hosted in the official [Chinook GitHub repository](https://github.com/lerocha/chinook-database).

In [ ]:
DB_PATH = Path("Chinook_Sqlite.sqlite")
DB_URL = (
    "https://github.com/lerocha/chinook-database/releases/download/"
    "v1.4.5/Chinook_Sqlite.sqlite"
)

if not DB_PATH.exists():
    print("Downloading Chinook SQLite database...")
    urllib.request.urlretrieve(DB_URL, DB_PATH)
    print(f"Downloaded to: {DB_PATH.resolve()}")
else:
    print(f"Using existing database: {DB_PATH.resolve()}")

# Open a connection. SQLite creates a new file if the path is wrong,
# so the existence check above helps prevent accidentally querying an empty DB.
connection = sqlite3.connect(DB_PATH)
print("SQLite connection established.")

## 3. Explore the database schema

First, list all tables. Then inspect each table’s columns and foreign-key relationships.

In [ ]:
tables_query = """
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
ORDER BY name;
"""
tables_df = pd.read_sql_query(tables_query, connection)
display(tables_df)
print(f"Number of tables: {len(tables_df)}")

In [ ]:
# Inspect columns for every table using SQLite's PRAGMA command.
schema_details = []
for table_name in tables_df["table_name"]:
    safe_name = table_name.replace('"', '""')
    columns = connection.execute(f'PRAGMA table_info("{safe_name}")').fetchall()
    for column in columns:
        schema_details.append({
            "table": table_name,
            "column": column[1],
            "data_type": column[2],
            "not_null": bool(column[3]),
            "primary_key": bool(column[5]),
        })

schema_df = pd.DataFrame(schema_details)
display(schema_df)

In [ ]:
# Display foreign keys so table relationships are explicit.
foreign_keys = []
for table_name in tables_df["table_name"]:
    safe_name = table_name.replace('"', '""')
    rows = connection.execute(f'PRAGMA foreign_key_list("{safe_name}")').fetchall()
    for row in rows:
        foreign_keys.append({
            "table": table_name,
            "column": row[3],
            "references_table": row[2],
            "references_column": row[4],
        })

relationships_df = pd.DataFrame(foreign_keys)
display(relationships_df)

### Key relationships

- `Artist` → `Album` → `Track`: artists release albums containing tracks.
- `Genre` and `MediaType` classify tracks.
- `Invoice` → `InvoiceLine` → `Track`: invoices contain purchased track line items.
- `Customer` → `Invoice`: customers place orders.
- `Employee` → `Customer`: `Customer.SupportRepId` identifies the assigned sales/support representative.
- `Playlist` ↔ `Track`: the many-to-many relationship is stored in `PlaylistTrack`.
- `Employee.ReportsTo` is a self-reference representing the reporting hierarchy.

## 4. Business question 1 — What are the 10 best-selling tracks?

**Metric:** total units sold, calculated by summing `InvoiceLine.Quantity` for each track. We join `Track` to `InvoiceLine` through `TrackId`.

In [ ]:
top_tracks_query = """
SELECT
    t.TrackId,
    t.Name AS track_name,
    ar.Name AS artist_name,
    SUM(il.Quantity) AS units_sold,
    ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS track_revenue
FROM InvoiceLine AS il
JOIN Track AS t ON t.TrackId = il.TrackId
LEFT JOIN Album AS al ON al.AlbumId = t.AlbumId
LEFT JOIN Artist AS ar ON ar.ArtistId = al.ArtistId
GROUP BY t.TrackId, t.Name, ar.Name
ORDER BY units_sold DESC, track_revenue DESC
LIMIT 10;
"""
top_tracks_df = pd.read_sql_query(top_tracks_query, connection)
display(top_tracks_df)

## 5. Business question 2 — Which country generates the most revenue?

**Metric:** sum of `Invoice.Total`, grouped by `BillingCountry`. This uses the invoice billing country rather than the customer’s current country.

In [ ]:
country_revenue_query = """
SELECT
    BillingCountry AS country,
    COUNT(InvoiceId) AS invoice_count,
    ROUND(SUM(Total), 2) AS total_revenue
FROM Invoice
GROUP BY BillingCountry
ORDER BY total_revenue DESC;
"""
country_revenue_df = pd.read_sql_query(country_revenue_query, connection)
display(country_revenue_df.head(10))

if not country_revenue_df.empty:
    top_country = country_revenue_df.iloc[0]
    print(
        f"Top revenue country: {top_country['country']} "
        f"(${top_country['total_revenue']:,.2f} across "
        f"{int(top_country['invoice_count'])} invoices)"
    )

## 6. Business question 3 — Who is the top-performing sales employee?

**Metric:** total invoice revenue from customers assigned to each employee. The relationship is `Employee.EmployeeId` → `Customer.SupportRepId` → `Invoice.CustomerId`. A `LEFT JOIN` keeps employees with no assigned customer sales in the result.

In [ ]:
employee_sales_query = """
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS employee_name,
    COUNT(DISTINCT c.CustomerId) AS assigned_customers,
    COUNT(DISTINCT i.InvoiceId) AS invoices_handled,
    ROUND(COALESCE(SUM(i.Total), 0), 2) AS total_revenue
FROM Employee AS e
LEFT JOIN Customer AS c ON c.SupportRepId = e.EmployeeId
LEFT JOIN Invoice AS i ON i.CustomerId = c.CustomerId
GROUP BY e.EmployeeId, e.FirstName, e.LastName
ORDER BY total_revenue DESC;
"""
employee_sales_df = pd.read_sql_query(employee_sales_query, connection)
display(employee_sales_df)

if not employee_sales_df.empty:
    top_employee = employee_sales_df.iloc[0]
    print(
        f"Top-performing sales employee: {top_employee['employee_name']} "
        f"(${top_employee['total_revenue']:,.2f} revenue)"
    )

## 7. Visualize the results

The chart below compares the 10 highest-revenue billing countries. Revenue is shown in the database’s currency (USD).

In [ ]:
top_countries = country_revenue_df.head(10).sort_values("total_revenue")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_countries["country"], top_countries["total_revenue"])
ax.set_title("Top 10 Countries by Invoice Revenue")
ax.set_xlabel("Revenue (USD)")
ax.set_ylabel("Billing country")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.grid(axis="x", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

## 8. Business findings summary

In [ ]:
print("BUSINESS FINDINGS")
print("-" * 50)
if not top_tracks_df.empty:
    best_track = top_tracks_df.iloc[0]
    print(f"Best-selling track: {best_track['track_name']} by {best_track['artist_name']} "
          f"({int(best_track['units_sold'])} units)")
if not country_revenue_df.empty:
    best_country = country_revenue_df.iloc[0]
    print(f"Highest-revenue country: {best_country['country']} "
          f"(${best_country['total_revenue']:,.2f})")
if not employee_sales_df.empty:
    best_rep = employee_sales_df.iloc[0]
    print(f"Top sales employee: {best_rep['employee_name']} "
          f"(${best_rep['total_revenue']:,.2f})")

## 9. Close the database connection

In [ ]:
connection.close()
print("Database connection closed.")

## Conclusion

This analysis used Python’s built-in `sqlite3` module to query a relational database, SQL joins and aggregations to answer business questions, Pandas to organize query outputs, and Matplotlib to visualize revenue. The definitions of “best-selling” and “top-performing” are explicitly based on units sold and invoice revenue, respectively.